In [3]:
"""
fundamentalistas_cvm.py
=======================
Extrai histórico de indicadores fundamentalistas (EBIT, EV, ROIC e outros)
a partir dos arquivos da CVM, dado uma lista de tickers.
 
Dependências:
    pip install pandas yfinance rapidfuzz requests
 
Uso:
    from fundamentalistas_cvm import get_fundamentals
 
    df = get_fundamentals(
        tickers=["WEGE3", "PETR4", "VALE3"],
        base_dados=base_dados,         # seu DataFrame da CVM
        consolidado=True,              # True = DF Consolidado, False = DF Individual
    )
"""
 
import warnings
import requests
import numpy as np
import pandas as pd
import yfinance as yf
from rapidfuzz import process, fuzz
 
warnings.filterwarnings("ignore")


In [4]:
# ---------------------------------------------------------------------------
# Códigos de conta CVM utilizados
# ---------------------------------------------------------------------------
# DRE
CONTA_EBIT           = "3.05"   # Resultado antes do Result. Financeiro e Tributos
CONTA_LL             = "3.11"   # Lucro/Prejuízo do Período
CONTA_IR_CSLL        = "3.08"   # IR e Contribuição Social
CONTA_RECEITA_LIQ    = "3.03"   # Resultado Bruto (proxy) | use 3.01 para Receita Bruta
 
# Balanço — Ativo
CONTA_CAIXA          = "1.01.01"  # Caixa e Equivalentes
CONTA_APLIC_CP       = "1.01.02"  # Aplicações Financeiras CP
CONTA_ATIVO_TOTAL    = "1"        # Ativo Total
 
# Balanço — Passivo
CONTA_DIVIDA_CP      = "2.01.04"  # Empréstimos e Financiamentos CP
CONTA_DIVIDA_LP      = "2.02.01"  # Empréstimos e Financiamentos LP
CONTA_PL             = "2.03"     # Patrimônio Líquido


In [5]:
# ---------------------------------------------------------------------------
# 1. Mapeamento Ticker → Nome CVM
# ---------------------------------------------------------------------------
 
def baixar_cadastro_cvm() -> pd.DataFrame:
    """
    Baixa o cadastro de empresas abertas da CVM.
    Contém: CD_CVM, DENOM_CIA, TCKR (ticker), CNPJ_CIA, entre outros.
    """
    url = "https://dados.cvm.gov.br/dados/CIA_ABERTA/CAD/DADOS/cad_cia_aberta.csv"
    try:
        df = pd.read_csv(url, sep=";", encoding="latin1")
        df.columns = df.columns.str.strip()
        # Normaliza ticker: remove sufixos como F (fracionário)
        if "TCKR" in df.columns:
            df["TCKR"] = df["TCKR"].str.strip().str.upper()
        return df
    except Exception as e:
        print(f"⚠ Não foi possível baixar cadastro CVM: {e}")
        return pd.DataFrame()
 
 
def _nome_via_yfinance(ticker: str) -> str | None:
    """Tenta obter o nome longo da empresa via yfinance."""
    try:
        info = yf.Ticker(f"{ticker}.SA").info
        return info.get("longName") or info.get("shortName")
    except Exception:
        return None
 
 
def _fuzzy_match(nome_busca: str, nomes_cvm: list[str], threshold: int = 75) -> str | None:
    """Fuzzy match do nome buscado contra os nomes da CVM."""
    if not nome_busca:
        return None
    result = process.extractOne(
        nome_busca.upper(),
        [n.upper() for n in nomes_cvm],
        scorer=fuzz.token_sort_ratio,
    )
    if result and result[1] >= threshold:
        # Retorna o nome original (não uppercased)
        idx = [n.upper() for n in nomes_cvm].index(result[0])
        return nomes_cvm[idx]
    return None


In [6]:
def mapear_tickers(tickers: list[str], base_dados: pd.DataFrame) -> dict[str, str]:
    """
    Retorna dicionário {ticker: DENOM_CIA} para os tickers fornecidos.
 
    Estratégia (em ordem):
      1. Cadastro CVM (coluna TCKR) — fonte mais confiável
      2. yfinance longName + fuzzy match contra DENOM_CIA do base_dados
      3. Aviso se não encontrado
    """
    nomes_cvm_unicos = base_dados["DENOM_CIA"].dropna().unique().tolist()
    cadastro = baixar_cadastro_cvm()
    mapeamento = {}
 
    for ticker in tickers:
        ticker_up = ticker.upper()
        nome_encontrado = None
 
        # --- Estratégia 1: cadastro CVM ---
        if not cadastro.empty and "TCKR" in cadastro.columns:
            match_cad = cadastro[cadastro["TCKR"] == ticker_up]
            if not match_cad.empty:
                nome_cad = match_cad.iloc[0]["DENOM_CIA"]
                # Confirma que existe no base_dados (pode ter variação de nome)
                nome_encontrado = _fuzzy_match(nome_cad, nomes_cvm_unicos, threshold=70)
                if nome_encontrado:
                    print(f"  ✓ {ticker_up:8s} → '{nome_encontrado}' (via cadastro CVM)")
 
        # --- Estratégia 2: yfinance + fuzzy ---
        if nome_encontrado is None:
            nome_yf = _nome_via_yfinance(ticker_up)
            if nome_yf:
                nome_encontrado = _fuzzy_match(nome_yf, nomes_cvm_unicos, threshold=70)
                if nome_encontrado:
                    print(f"  ✓ {ticker_up:8s} → '{nome_encontrado}' (via yfinance + fuzzy)")
 
        if nome_encontrado is None:
            print(f"  ✗ {ticker_up:8s} → NÃO ENCONTRADO. Verifique o ticker ou adicione mapeamento manual.")
 
        mapeamento[ticker_up] = nome_encontrado
 
    return mapeamento


In [7]:
# ---------------------------------------------------------------------------
# 2. Extração de contas do base_dados
# ---------------------------------------------------------------------------
 
def _extrair_conta(df: pd.DataFrame, cd_conta: str, nome_col: str) -> pd.DataFrame:
    """
    Filtra linhas pelo CD_CONTA exato e pivota para uma coluna com nome_col.
    Retorna DataFrame com [DENOM_CIA, DT_FIM_EXERC, nome_col].
    """
    sub = df[df["CD_CONTA"] == cd_conta][
        ["DENOM_CIA", "DT_FIM_EXERC", "VL_CONTA"]
    ].copy()
    sub = sub.rename(columns={"VL_CONTA": nome_col})
    return sub
 
 
def _extrair_conta_prefixo(df: pd.DataFrame, prefixo: str, nome_col: str) -> pd.DataFrame:
    """
    Para contas como Ativo Total (CD_CONTA == '1') que podem ter variações.
    Pega a conta cujo CD_CONTA bate exato com o prefixo.
    """
    return _extrair_conta(df, prefixo, nome_col)
 


In [8]:
# ---------------------------------------------------------------------------
# 3. Preço histórico e shares via yfinance (para Market Cap e EV)
# ---------------------------------------------------------------------------
 
def _buscar_preco_shares(ticker: str, datas: list) -> pd.DataFrame:
    """
    Busca o preço de fechamento ajustado e quantidade de ações via yfinance
    para as datas dos demonstrativos.
 
    Retorna DataFrame com [DT_FIM_EXERC, preco_fechamento, shares_outstanding].
    """
    try:
        t = yf.Ticker(f"{ticker}.SA")
        info = t.info
 
        # Shares: prefere fast_info, cai para info
        shares = (
            getattr(t.fast_info, "shares", None)
            or info.get("sharesOutstanding")
        )
 
        # Histórico de preços — pega intervalo que cobre todas as datas
        datas_dt = pd.to_datetime(datas)
        start = datas_dt.min() - pd.DateOffset(days=10)
        end   = datas_dt.max() + pd.DateOffset(days=10)
 
        hist = t.history(start=start, end=end, interval="1mo", auto_adjust=True)
        if hist.empty:
            return pd.DataFrame()
 
        hist = hist[["Close"]].reset_index()
        hist["Date"] = pd.to_datetime(hist["Date"]).dt.tz_localize(None)
        hist["shares"] = shares
 
        rows = []
        for data in datas_dt:
            # Preço mais próximo da data de fim do exercício
            diff = (hist["Date"] - data).abs()
            idx = diff.idxmin()
            rows.append({
                "DT_FIM_EXERC": data,
                "preco_fechamento": hist.loc[idx, "Close"],
                "shares_outstanding": shares,
            })
 
        return pd.DataFrame(rows)
 
    except Exception as e:
        print(f"    ⚠ yfinance ({ticker}): {e}")
        return pd.DataFrame()


In [9]:
# ---------------------------------------------------------------------------
# 4. Cálculo dos indicadores
# ---------------------------------------------------------------------------
 
def _calcular_indicadores(df: pd.DataFrame, ticker: str) -> pd.DataFrame:
    """
    Calcula EBIT, Net Debt, Market Cap, EV e ROIC a partir das colunas extraídas.
 
    Entradas esperadas (colunas):
        ebit, caixa, aplic_cp, divida_cp, divida_lp, pl, ativo_total,
        ir_csll, ll, preco_fechamento, shares_outstanding
    """
    df = df.copy()
 
    # --- Dívida Líquida ---
    df["disponibilidades"] = df.get("caixa", 0).fillna(0) + df.get("aplic_cp", 0).fillna(0)
    df["divida_bruta"]     = df.get("divida_cp", 0).fillna(0) + df.get("divida_lp", 0).fillna(0)
    df["divida_liquida"]   = df["divida_bruta"] - df["disponibilidades"]
 
    # --- Market Cap ---
    if "preco_fechamento" in df.columns and "shares_outstanding" in df.columns:
        df["market_cap"] = df["preco_fechamento"] * df["shares_outstanding"].fillna(0)
    else:
        df["market_cap"] = np.nan
 
    # --- EV ---
    df["ev"] = df["market_cap"] + df["divida_liquida"]
 
    # --- Alíquota Efetiva de IR/CSLL ---
    # IR e CSLL costumam vir negativos na DRE (despesa)
    df["ir_csll"]  = df.get("ir_csll", pd.Series(0, index=df.index)).fillna(0).abs()
    df["ebt"]      = df.get("ebit", np.nan) + df.get("resultado_financeiro", 0).fillna(0)
 
    # Evita divisão por zero
    with np.errstate(divide="ignore", invalid="ignore"):
        df["aliquota_efetiva"] = np.where(
            df.get("ll", pd.Series(np.nan, index=df.index)).abs() > 0,
            df["ir_csll"] / (df.get("ll", pd.Series(np.nan, index=df.index)).abs() + df["ir_csll"]),
            0.34,   # alíquota padrão BR se não conseguir calcular
        )
    df["aliquota_efetiva"] = df["aliquota_efetiva"].clip(0, 0.50)
 
    # --- NOPAT ---
    df["nopat"] = df.get("ebit", pd.Series(np.nan, index=df.index)) * (1 - df["aliquota_efetiva"])
 
    # --- Capital Investido ---
    df["capital_investido"] = df.get("pl", pd.Series(np.nan, index=df.index)).fillna(0) + df["divida_liquida"]
 
    # --- ROIC ---
    with np.errstate(divide="ignore", invalid="ignore"):
        df["roic"] = np.where(
            df["capital_investido"] != 0,
            df["nopat"] / df["capital_investido"],
            np.nan,
        )
 
    # --- EV/EBIT ---
    with np.errstate(divide="ignore", invalid="ignore"):
        df["ev_ebit"] = np.where(
            df.get("ebit", pd.Series(np.nan, index=df.index)) != 0,
            df["ev"] / df.get("ebit", pd.Series(np.nan, index=df.index)),
            np.nan,
        )
 
    df.insert(0, "ticker", ticker)
    return df


In [10]:
# ---------------------------------------------------------------------------
# 5. Pipeline principal
# ---------------------------------------------------------------------------
 
def get_fundamentals(
    tickers: list[str],
    base_dados: pd.DataFrame,
    consolidado: bool = True,
    mapeamento_manual: dict[str, str] | None = None,
    output_csv: str | None = "fundamentalistas.csv",
    escala_moeda: str = "MIL",
) -> pd.DataFrame:
    """
    Extrai histórico fundamentalista para uma lista de tickers.
 
    Parâmetros
    ----------
    tickers           : lista de tickers (ex: ["WEGE3", "PETR4"])
    base_dados        : DataFrame da CVM já carregado
    consolidado       : True = DF Consolidado | False = DF Individual
    mapeamento_manual : dict extra {ticker: DENOM_CIA} para casos difíceis
                        ex: {"ITUB4": "ITAÚ UNIBANCO HOLDING S.A."}
    output_csv        : caminho para salvar o CSV (None = não salva)
    escala_moeda      : "MIL" ou "UNIDADE" — normaliza para R$ (divide por 1000 se MIL)
 
    Retorna
    -------
    DataFrame com colunas:
        ticker, DENOM_CIA, DT_FIM_EXERC,
        ebit, nopat, divida_bruta, divida_liquida, disponibilidades,
        pl, ativo_total, market_cap, ev, capital_investido,
        roic, ev_ebit, aliquota_efetiva
    """
    print("=" * 60)
    print("Mapeando tickers → nomes CVM...")
    print("=" * 60)
 
    mapeamento = mapear_tickers(tickers, base_dados)
 
    if mapeamento_manual:
        for tk, nome in mapeamento_manual.items():
            mapeamento[tk.upper()] = nome
            print(f"  ✎ {tk.upper():8s} → '{nome}' (mapeamento manual)")
 
    # Filtra apenas empresas encontradas
    tickers_validos = {tk: nome for tk, nome in mapeamento.items() if nome}
    if not tickers_validos:
        print("Nenhum ticker mapeado com sucesso.")
        return pd.DataFrame()
 
    # --- Prepara base filtrada ---
    cond_ind = "DF Consolidado" if consolidado else "DF Individual"
 
    # Normaliza escala de moeda
    fator = 1000 if escala_moeda == "MIL" else 1
 
    base_filtrada = base_dados[
        (base_dados["DENOM_CIA"].isin(tickers_validos.values())) &
        (base_dados["cond_ind"] == cond_ind)
    ].copy()
 
    base_filtrada["DT_FIM_EXERC"] = pd.to_datetime(base_filtrada["DT_FIM_EXERC"])
    base_filtrada["VL_CONTA"] = pd.to_numeric(base_filtrada["VL_CONTA"], errors="coerce") * fator
 
    all_frames = []
 
    for ticker, nome_cvm in tickers_validos.items():
        print(f"\nProcessando {ticker} ({nome_cvm})...")
 
        emp = base_filtrada[base_filtrada["DENOM_CIA"] == nome_cvm].copy()
        if emp.empty:
            print(f"  ✗ Sem dados no base_dados para {nome_cvm}")
            continue
 
        datas = sorted(emp["DT_FIM_EXERC"].unique())
 
        # Extrai cada conta
        def extrair(cd, col):
            return _extrair_conta(emp, cd, col).set_index("DT_FIM_EXERC")[col]
 
        series = {
            "ebit":       extrair(CONTA_EBIT,        "ebit"),
            "ll":         extrair(CONTA_LL,           "ll"),
            "ir_csll":    extrair(CONTA_IR_CSLL,      "ir_csll"),
            "caixa":      extrair(CONTA_CAIXA,        "caixa"),
            "aplic_cp":   extrair(CONTA_APLIC_CP,     "aplic_cp"),
            "divida_cp":  extrair(CONTA_DIVIDA_CP,    "divida_cp"),
            "divida_lp":  extrair(CONTA_DIVIDA_LP,    "divida_lp"),
            "pl":         extrair(CONTA_PL,           "pl"),
            "ativo_total":extrair(CONTA_ATIVO_TOTAL,  "ativo_total"),
        }
 
        df_ind = pd.DataFrame(series)
        df_ind.index.name = "DT_FIM_EXERC"
        df_ind = df_ind.reset_index()
        df_ind["DENOM_CIA"] = nome_cvm
 
        # Busca preço histórico
        print(f"  → Buscando preços via yfinance...")
        df_preco = _buscar_preco_shares(ticker, datas)
 
        if not df_preco.empty:
            df_preco["DT_FIM_EXERC"] = pd.to_datetime(df_preco["DT_FIM_EXERC"])
            df_ind = df_ind.merge(df_preco, on="DT_FIM_EXERC", how="left")
        else:
            df_ind["preco_fechamento"]   = np.nan
            df_ind["shares_outstanding"] = np.nan
 
        # Calcula indicadores
        df_calc = _calcular_indicadores(df_ind, ticker)
        all_frames.append(df_calc)
        print(f"  ✓ {len(df_calc)} períodos processados")
 
    if not all_frames:
        print("\nNenhum dado processado.")
        return pd.DataFrame()
 
    resultado = pd.concat(all_frames, ignore_index=True)
 
    # Seleciona e ordena colunas finais
    cols_finais = [
        "ticker", "DENOM_CIA", "DT_FIM_EXERC",
        "ebit", "nopat",
        "disponibilidades", "divida_bruta", "divida_liquida",
        "pl", "ativo_total",
        "market_cap", "ev",
        "capital_investido", "roic", "ev_ebit",
        "aliquota_efetiva", "ll",
    ]
    cols_presentes = [c for c in cols_finais if c in resultado.columns]
    resultado = resultado[cols_presentes].sort_values(["ticker", "DT_FIM_EXERC"])
 
    if output_csv:
        resultado.to_csv(output_csv, index=False, encoding="utf-8-sig")
        print(f"\n✅ Salvo: {output_csv}")
 
    print(f"\n{'='*60}")
    print(f"Concluído: {resultado['ticker'].nunique()} empresas | "
          f"{len(resultado)} períodos")
 
    return resultado


In [18]:
df_fund = get_fundamentals(
        tickers=["WEGE3", "PETR4", "VALE3", "ITUB4"],
        base_dados=pd.read_csv('dados_cvm/CVM.csv', sep=',', encoding='latin1'),
        consolidado=True,
        # Casos onde o fuzzy match falha — adicione manualmente:
        mapeamento_manual={
            # "ITUB4": "ITAÚ UNIBANCO HOLDING S.A.",
        },
        output_csv="fundamentalistas.csv",
        escala_moeda="MIL",   # arquivos CVM padrão vêm em milhares
    )
 
print(df_fund[["ticker", "DT_FIM_EXERC", "ebit", "ev", "roic", "ev_ebit"]].to_string())


Mapeando tickers → nomes CVM...
  ✓ WEGE3    → 'WEG S.A.' (via yfinance + fuzzy)
  ✓ PETR4    → 'PETROLEO BRASILEIRO S.A. PETROBRAS' (via yfinance + fuzzy)
  ✓ VALE3    → 'VALE S.A.' (via yfinance + fuzzy)
  ✓ ITUB4    → 'ITAU UNIBANCO HOLDING S.A.' (via yfinance + fuzzy)

Processando WEGE3 (WEG S.A.)...
  → Buscando preços via yfinance...


AttributeError: 'int' object has no attribute 'fillna'